# Foodbank Agent — Evaluation Notebook

Run this on Google Colab to verify the full pipeline with real credentials.

**Sections:**
1. Setup — clone repo, install deps, load secrets
2. Sheet import — load both Google Sheets, inspect shape and data quality
3. LLM connectivity — one quick call to confirm the API key and model work
4. Evaluations — 4 queries where pandas computes the ground truth and the LLM answer is checked (PASS / FAIL)

---
**Add these secrets in Colab → 🔑 Secrets (left sidebar) before running:**

| Secret | Where to get it |
|---|---|
| `GROQ_API_KEY` | console.groq.com/keys |
| `CEREBRAS_API_KEY` | cloud.cerebras.ai → API Keys |
| `GOOGLE_CREDENTIALS_B64` | Render dashboard → env vars |
| `SHEET_ID` | Render dashboard → env vars |
| `LOGIN_SHEET_ID` | Render dashboard → env vars |

---
## 0 · Setup
Clone the repo and install dependencies. Run once per session.

In [ ]:
import os

REPO_DIR = "/content/Foodbank_agent"

if not os.path.exists(REPO_DIR):
    !git clone -b pandas-code-gen https://github.com/spykeshivam/Foodbank_agent.git
else:
    !cd {REPO_DIR} && git pull origin pandas-code-gen

%cd {REPO_DIR}
print("Repo ready:", os.getcwd())

In [ ]:
!pip install -q openai pandas plotly openpyxl python-dateutil gspread python-dotenv
print("Dependencies installed")

In [ ]:
from google.colab import userdata

REQUIRED = [
    "GROQ_API_KEY",
    "CEREBRAS_API_KEY",
    "GOOGLE_CREDENTIALS_B64",
    "SHEET_ID",
    "LOGIN_SHEET_ID",
]

missing = []
for key in REQUIRED:
    try:
        os.environ[key] = userdata.get(key)
        print(f"  OK  {key}")
    except Exception:
        missing.append(key)
        print(f"  !!  {key}  <- missing")

if missing:
    print("\nAdd missing secrets in Colab -> Secrets, then re-run this cell.")
else:
    print("\nAll secrets loaded.")

In [ ]:
import re
import sys

import pandas as pd

sys.path.insert(0, REPO_DIR)

from agent import PROVIDERS, run_query
from sheets import fetch_logins, fetch_registrations

PROVIDER = "Groq"  # change to "Cerebras" to test the other provider
print(f"Using provider: {PROVIDER} — model: {PROVIDERS[PROVIDER]['model']}")

---
## 1 · Sheet Import

In [ ]:
print("Loading registrations...")
registrations = pd.DataFrame(fetch_registrations())
print(f"  {len(registrations):,} rows x {len(registrations.columns)} columns")

print("Loading logins...")
logins = pd.DataFrame(fetch_logins())
print(f"  {len(logins):,} rows x {len(logins.columns)} columns")

In [ ]:
print("Registrations columns:", list(registrations.columns))
registrations.head(3)

In [ ]:
print("Logins columns:", list(logins.columns))
logins.head(3)

In [ ]:
# Quick data-quality check
print("Sex breakdown:")
print(registrations["Sex"].value_counts().to_string())
print("\nTop dietary requirements:")
print(registrations["Dietary Requirements"].value_counts().head(8).to_string())
print("\nLogins by day:")
print(logins["Day"].value_counts().to_string())

---
## 2 · LLM Connectivity Check

One call to confirm the API key and model name are both working.
If this errors, the model name or key is wrong — fix those before running evaluations.

In [ ]:
probe = run_query(
    "How many rows are in the registrations sheet?",
    registrations,
    logins,
    provider=PROVIDER,
)

if probe.error:
    print("FAIL — agent error:")
    print(probe.error)
else:
    print(f"OK — {PROVIDER} ({PROVIDERS[PROVIDER]['model']}) is reachable")
    print("Response:", probe.text)

---
## 3 · Evaluations

Each evaluation:
- Computes the **ground-truth** answer directly with pandas
- Asks the **LLM** the same question
- Checks whether the expected number appears in the response → **PASS / FAIL**

This is the key test: if the LLM gets the number wrong, either the code it generated is buggy or the model is hallucinating.

In [ ]:
def evaluate(label: str, question: str, expected: int) -> bool:
    print(f"\n--- {label} ---")
    print(f"  Expected : {expected:,}")
    result = run_query(question, registrations, logins, provider=PROVIDER)
    if result.error:
        print(f"  FAIL (agent error): {result.error[:120]}")
        return False
    # Strip thousand separators then look for the number anywhere in the text
    clean_text = re.sub(r"[,_]", "", result.text)
    passed = str(expected) in clean_text
    print(f"  {'PASS' if passed else 'FAIL'}")
    print(f"  LLM said : {result.text.strip()}")
    return passed

In [ ]:
# Eval 1 — Total registered users
evaluate(
    "Total registered users",
    "How many people are registered in total?",
    len(registrations),
)

In [ ]:
# Eval 2 — Halal dietary requirements
halal_count = int(
    registrations["Dietary Requirements"].str.contains("Halal", case=False, na=False).sum()
)
evaluate(
    "Halal dietary requirements",
    "How many people have Halal dietary requirements?",
    halal_count,
)

In [ ]:
# Eval 3 — Total login visits
evaluate(
    "Total login visits",
    "How many total visits have been logged?",
    len(logins),
)

In [ ]:
# Eval 4 — Male users
male_count = int(registrations["Sex"].str.strip().str.lower().eq("male").sum())
evaluate(
    "Male users",
    "How many male users are registered?",
    male_count,
)

---
## Summary — run all 4 at once

In [ ]:
import time

evals = [
    (
        "Total registrations",
        "How many people are registered in total?",
        len(registrations),
    ),
    (
        "Halal requirements",
        "How many people have Halal dietary requirements?",
        int(registrations["Dietary Requirements"].str.contains("Halal", case=False, na=False).sum()),
    ),
    (
        "Total logins",
        "How many total visits have been logged?",
        len(logins),
    ),
    (
        "Male users",
        "How many male users are registered?",
        int(registrations["Sex"].str.strip().str.lower().eq("male").sum()),
    ),
]

results = []
for i, (label, question, expected) in enumerate(evals):
    if i > 0:
        time.sleep(5)  # stay within free-tier rate limits
    passed = evaluate(label, question, expected)
    results.append((label, expected, passed))

print("\n" + "=" * 45)
print(f"RESULTS  ({PROVIDER} — {PROVIDERS[PROVIDER]['model']})")
print("=" * 45)
for label, expected, passed in results:
    icon = "PASS" if passed else "FAIL"
    print(f"  {icon}  {label:<25}  expected {expected:,}")
total = sum(p for _, _, p in results)
print(f"\n  {total}/{len(results)} passed")